In [36]:
# Importing the libraries
import pandas as pd 
import numpy as np
import os
from tqdm.notebook import tqdm

# **TASK 1**

In [39]:
#fetching the important data
all_files = os.listdir("SampleData")
cp_files=[]
for file in all_files:
    if str(file).find("closePosition")==-1:
        pass
    else:
        cp_files.append(file)

In [190]:
# extracting and merging the data files
main_df = pd.DataFrame()
for i, file in tqdm(enumerate(cp_files),total=len(cp_files)):
    df = pd.read_csv(f"SampleData/{file}",parse_dates=True)
    df = df[['Key', 'ExitTime', 'Symbol', 'EntryPrice', 'Quantity', 'Pnl']]
    df['Date'] = pd.to_datetime(df.ExitTime).dt.date
    main_df = pd.concat([main_df,df])

  0%|          | 0/1110 [00:00<?, ?it/s]

In [191]:
main_df = main_df.drop_duplicates(keep="first")

In [192]:
main_df_1 =main_df.copy()

# **Task 2**

In [193]:
print(f"The total number of trades in the final dataframe {len(main_df)}")
print(f"The number of unique dates in the Date column are {len(np.unique(main_df.Date))} ")
print(f"Average trades are {len(main_df)/len(np.unique(main_df.Date)):.3f}")
print(f"Total pnl: {np.sum(main_df.Pnl):.3f}")
print(f"Number of Profitable Trades {main_df[main_df.Pnl>0].Pnl.count()}")
print(f"Number of loss Trades {main_df[main_df.Pnl<=0].Pnl.count()}")

The total number of trades in the final dataframe 1675
The number of unique dates in the Date column are 141 
Average trades are 11.879
Total pnl: 635766.250
Number of Profitable Trades 558
Number of loss Trades 1117


In [194]:
combined_stats = {"Total trades":len(main_df),
                 "Unique days":len(np.unique(main_df.Date)),
                 "Average trades": len(main_df)/len(np.unique(main_df.Date)),
                 "Total pnl": np.sum(main_df.Pnl),
                  "Profit Trades": main_df[main_df.Pnl>0].Pnl.count(),
                  "Loss Trades": main_df[main_df.Pnl<=0].Pnl.count()
                 }

with open('combined_stats.txt','w') as data: 
      data.write(str(combined_stats))

# **Task 3** 

In [195]:
main_df = main_df.sort_values("ExitTime",ascending=True)
score = []

for index, row in main_df.iterrows():
    if row['Pnl']>0:
        score.append(1)
    else:
        score.append(0)
main_df['score'] = score

In [196]:
# Equating trade to the previous type of trade
main_df["streak"] = main_df["score"].ne(main_df["score"].shift())

# Adding all Yeses/Trues to group them later
main_df["streak_number"] = main_df["streak"].cumsum()

# Grouping the streaks and finding the count
main_df["streak_count"] = main_df.groupby("streak_number").cumcount().add(1)

main_df.reset_index(inplace=True)

In [207]:
topn_streaks = main_df.groupby("streak_number")['Pnl'].sum()\
                .reset_index().sort_values("Pnl",ascending=False)

In [321]:
def top_n_streaks():
    n = input()
    win_streak = pd.DataFrame()
    loss_streak = pd.DataFrame()
    
    win_numbers = topn_streaks.streak_number.iloc[:int(n)].values
    loss_numbers = topn_streaks.streak_number.iloc[-int(n):].values
    loss_numbers = np.flip(loss_numbers)
    
    total_trades, date ,pnl_of_streaks= [], [], []
    for number in win_numbers:
        df=pd.DataFrame()
        temp_df = main_df.loc[main_df.streak_number==number].reset_index()
        total_trades.append(len(temp_df))
        date.append(f"{temp_df.iloc[0].ExitTime} to {temp_df.iloc[-1].ExitTime}")
        pnl_of_streaks.append(temp_df.Pnl.sum())
    win_streak["total_trades"]=total_trades
    win_streak["duration"] = date
    win_streak['pnl_of_streaks'] = pnl_of_streaks
    win_streak.to_csv("win.csv")
    
    total_trades,date, pnl_of_streaks= [], [], []
    for number in loss_numbers:
        df=pd.DataFrame()
        temp_df = main_df.loc[main_df.streak_number==number].reset_index()
        total_trades.append(len(temp_df))
        date.append(f"{temp_df.iloc[0].ExitTime} to {temp_df.iloc[-1].ExitTime}")
        pnl_of_streaks.append(temp_df.Pnl.sum())
    loss_streak["total_trades"]=total_trades
    loss_streak["duration"] = date
    loss_streak['pnl_of_streaks'] = pnl_of_streaks
    loss_streak.to_csv("win.csv")
    
    return win_streak, loss_streak

In [322]:
a,b = top_n_streaks()

3


In [324]:
a

,total_trades,duration,pnl_of_streaks
0,6,2021-02-01 11:29:00 to 2021-02-02 09:27:00,27826.25
1,7,2020-12-09 13:56:00 to 2020-12-10 10:26:00,22266.25
2,6,2020-11-17 11:16:00 to 2020-11-17 14:56:00,19653.75


In [325]:
b

,total_trades,duration,pnl_of_streaks
0,10,2021-01-29 14:12:00 to 2021-02-01 10:44:00,-15396.25
1,11,2021-02-18 09:26:00 to 2021-02-18 11:29:00,-12970.00
2,15,2020-08-06 09:25:00 to 2020-08-06 12:26:00,-12872.50
